# E-Commerce Recommendation System

End-to-end demo using **Ray Data → Ray Train → Ray Data → Ray Serve** on Anyscale.

```
┌──────────────────────────────────────────────────────────────────────┐
│  Product Recommender                                                 │
│                                                                      │
│  ▼  Stage 1 — Ray Data                                               │
│       Preprocess images (resize, normalise)                          │
│       Clean text (descriptions + category)                           │
│  │                                                                   │
│  ▼  Stage 2 — Ray Train (TorchTrainer)                               │
│       Fine-tune all-MiniLM-L6-v2 with contrastive loss               │
│       Save fine-tuned model checkpoint                               │
│  │                                                                   │
│  ▼  Stage 3 — Ray Data (Batch Embedding)                             │
│       Embed entire product catalog with fine-tuned model             │
│       Save embeddings + metadata for serving                         │
│  │                                                                   │
│  ▼  Stage 4 — Ray Serve                                              │
│       POST /recommend  {image_base64}                                │
│       ImageToText  (BLIP-base)  →  caption                           │
│       ProductRecommender  (MiniLM)  →  top-5 products                │
│  │                                                                   │
│  ▼  Streamlit UI                                                     │
│       Upload image                                                   │
│                                                                      │
└──────────────────────────────────────────────────────────────────────┘
```

**Models used**
| Stage | Model | Size | Device |
|-------|-------|------|--------|
| Train | `sentence-transformers/all-MiniLM-L6-v2` | 22 M params | CPU |
| Embed | fine-tuned MiniLM | 22 M params | CPU |
| Serve | `Salesforce/blip-image-captioning-base` | 224 M params | CPU |
| Serve | fine-tuned MiniLM | 22 M params | CPU |

Everything runs on **CPU** and completes in a few minutes.

## 0 — Setup

In [ ]:
# Install dependencies (skip if already installed)
!pip install -q -r setup/requirements.txt

### Peek at the product images

In [ ]:
from utils import PRODUCTS, CATEGORIES, get_product_image
from utils.viz import plot_category_samples

plot_category_samples(PRODUCTS, CATEGORIES, get_product_image)

---
## Stage 1 — Catalog Generation

Here we expand the base 30-product catalog to 1,000 products and add `text_clean` (strip / lowercase / collapse whitespace on `training_text`) directly in memory — no Parquet intermediate needed.

In [ ]:
import os

from utils import (
    PRODUCTS,
    attach_clean_text,
    expand_catalog,
    generate_catalog,
    init_ray,
)

init_ray()

raw_dir = os.path.abspath("data/raw")

products = expand_catalog(PRODUCTS, target_size=1000)
print(f"Catalog size : {len(products)} products")
print(f"Categories   : {sorted(set(p['category'] for p in products))}")

catalog_records = attach_clean_text(generate_catalog(products=products, output_dir=raw_dir))
print(f"Catalog records : {len(catalog_records)}")


---
## Stage 2 — Ray Train: Embedding Fine-Tuning

The starter model (`all-MiniLM-L6-v2`) learned from general web text, not from our product categories. That makes it easy for a fuzzy query to land near the wrong kind of item in the embedding space.

**Fine-tuning** trains the model with **contrastive loss**: same-category products move closer, different-category products move apart, so search and recommendations follow category boundaries better.

We train on **100 products (20 per category)** so each category is represented and runs stay short. The t-SNE plots below show embeddings before vs after fine-tuning.

In [ ]:
import os
import tempfile

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from ray.train import Checkpoint, get_checkpoint, get_context, report

from utils import resolve_artifact_paths, sample_per_category
from utils.training import ContrastivePairDataset, forward_embeddings

# Where to read/write fine-tuned model + embeddings + train results.
paths = resolve_artifact_paths()
MODEL_OUTPUT_DIR = paths["model_dir"]
EMBEDDINGS_PATH  = paths["embeddings_path"]
METADATA_PATH    = paths["metadata_path"]
TRAIN_RESULT_DIR = paths["train_result_dir"]

# Class-balanced 100-record sample (20 per category) keeps the demo short
# while giving contrastive loss positives in every category.
TRAIN_PER_CATEGORY = 20
train_records = sample_per_category(
    catalog_records, n_per_category=TRAIN_PER_CATEGORY, seed=42
)


def train_loop_per_worker(config: dict) -> None:
    """Runs on every Ray Train worker. Ray Train APIs in use:
       • get_context().get_world_rank()      — which worker am I in the group?
       • get_checkpoint()                    — resume after a restart
       • report(metrics, checkpoint=...)     — stream metrics + artifacts to the driver
       • Checkpoint.from_directory(path)     — wrap a folder as a Ray checkpoint
    """
    from sentence_transformers import SentenceTransformer

    rank   = get_context().get_world_rank()
    device = "cuda" if torch.cuda.is_available() else "cpu"

    def collate(batch):
        a, b, lab = zip(*batch)
        return list(a), list(b), torch.stack(lab)

    loader = DataLoader(
        ContrastivePairDataset(config["records"], seed=config["seed"]),
        batch_size=config["batch_size"], shuffle=True, collate_fn=collate,
    )
    model     = SentenceTransformer(config["base_model"], device=device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["lr"], weight_decay=0.01)

    # Resume from the last good checkpoint if Ray Train restarted this worker.
    start_epoch = 0
    if (ckpt := get_checkpoint()) is not None:
        with ckpt.as_directory() as d:
            start_epoch = torch.load(
                os.path.join(d, "meta.pt"), weights_only=False
            )["epoch"] + 1
            model = SentenceTransformer(d, device=device)

    for epoch in range(start_epoch, config["epochs"]):
        model.train()
        loss_sum, n = 0.0, 0
        for a, b, labels in loader:
            ea = forward_embeddings(model, a, device)
            eb = forward_embeddings(model, b, device)
            loss = F.mse_loss(F.cosine_similarity(ea, eb), labels.to(device))
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            loss_sum += loss.item(); n += 1

        # Rank 0 packages the latest weights as a Ray Checkpoint; every worker
        # then calls report() so Ray treats the step as finished and persists
        # the checkpoint+metric per the trainer's CheckpointConfig.
        with tempfile.TemporaryDirectory() as tmp:
            ckpt_out = None
            if rank == 0:
                model.save(tmp)
                torch.save({"epoch": epoch}, os.path.join(tmp, "meta.pt"))
                ckpt_out = Checkpoint.from_directory(tmp)
            report(
                {"epoch": epoch, "train_loss": loss_sum / max(1, n)},
                checkpoint=ckpt_out,
            )


print(f"train_loop_per_worker defined")
print(f"  records       : {len(train_records)} ({TRAIN_PER_CATEGORY} per category)")
print(f"  device        : {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"  output dir    : {MODEL_OUTPUT_DIR}")

In [ ]:
from ray.train import CheckpointConfig, FailureConfig, RunConfig, ScalingConfig
from ray.train.torch import TorchTrainer

# `TorchTrainer` drives `train_loop_per_worker` across the worker group and
# orchestrates checkpoints, retries, and result aggregation.
#   ScalingConfig    → how many workers, whether to use GPUs
#   RunConfig        → where artifacts go on disk + checkpoint/failure policy
#   CheckpointConfig → which checkpoints to keep (here: best 2 by train_loss)
#   FailureConfig    → how many automatic retries on worker failure
scaling = ScalingConfig(num_workers=1, use_gpu=torch.cuda.is_available())

trainer = TorchTrainer(
    train_loop_per_worker=train_loop_per_worker,
    train_loop_config={
        "base_model": "sentence-transformers/all-MiniLM-L6-v2",
        "epochs": 2,
        "batch_size": 8,
        "lr": 2e-5,
        "seed": 42,
        "records": train_records,
    },
    scaling_config=scaling,
    run_config=RunConfig(
        name="ecomm_embedding_finetune",
        storage_path=os.path.abspath(TRAIN_RESULT_DIR),
        checkpoint_config=CheckpointConfig(
            num_to_keep=2,
            checkpoint_score_attribute="train_loss",
            checkpoint_score_order="min",
        ),
        failure_config=FailureConfig(max_failures=1),
    ),
)
print("TorchTrainer configured")
print(f"  workers   : {scaling.num_workers}   use_gpu: {scaling.use_gpu}")
print(f"  storage   : {TRAIN_RESULT_DIR}")
print(f"  keep      : best 2 checkpoints by min(train_loss)")

In [ ]:
# Run the fine-tune and persist the best checkpoint in a ready-to-serve layout.
from pathlib import Path
from sentence_transformers import SentenceTransformer

result = trainer.fit()
print(result.metrics_dataframe[["epoch", "train_loss"]].to_string(index=False))

# Ray Train sorts `result.best_checkpoints` by the metric+order we set on
# CheckpointConfig (min train_loss here), so [0][0] is the lowest-loss model.
# `.as_directory()` materialises it locally — even if Ray persisted it to
# remote storage like S3 — so we can reload + resave in HF format.
best_ckpt = result.best_checkpoints[0][0]
Path(MODEL_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
with best_ckpt.as_directory() as ckpt_dir:
    SentenceTransformer(ckpt_dir).save(MODEL_OUTPUT_DIR)
print(f"Model saved → {MODEL_OUTPUT_DIR}")

# Sanity check: load the saved artifact and encode something. If this fails,
# we find out here instead of inside a Ray Data actor 30 seconds later.
_probe = SentenceTransformer(MODEL_OUTPUT_DIR).encode(["wireless headphones"])
print(f"Saved model encodes OK → embedding shape {_probe.shape}")

In [ ]:
from utils.viz import plot_training_loss

plot_training_loss(result.metrics_dataframe)

---
## Stage 3 — Ray Data: Batch Embedding

**The problem this solves:** embedding 1,000 products serially with a model loaded once per process is slow. Worse, at 1M products you can't fit the whole catalog in memory on one machine.

Ray Data solves both: each actor loads the fine-tuned model **once**, then processes a steady stream of batches. Workers run in parallel, and Ray's streaming execution means only a slice of the catalog is in memory at any time. The same code scales from 1,000 to 100M products by adjusting the `ActorPoolStrategy` size.

In [ ]:
import ray
import numpy as np
from sentence_transformers import SentenceTransformer

# A stateful Ray Data actor: load the fine-tuned model **once** at construction,
# then encode many batches of text. Replicated across the actor pool below so
# the model-load cost (the expensive part) amortises across the whole catalog.
class ProductEmbedder:
    def __init__(self, model_dir: str):
        self.model = SentenceTransformer(model_dir)

    def __call__(self, batch: dict) -> dict:
        embs = self.model.encode(
            batch["text_clean"].tolist(),
            convert_to_numpy=True,
            show_progress_bar=False,
        )
        return {
            "product_id": batch["product_id"],
            "name":       batch["name"],
            "category":   batch["category"],
            "embedding":  list(embs),
        }


ds = ray.data.from_items(catalog_records).select_columns(
    ["product_id", "name", "category", "text_clean"]
)

# Bumping ActorPoolStrategy(size=...) is the only knob needed to scale this
# from 1k to 100M products. Ray streams batches through the pool, so the
# pipeline never needs to hold the whole catalog in memory at once.
rows = ds.map_batches(
    ProductEmbedder,
    fn_constructor_kwargs={"model_dir": MODEL_OUTPUT_DIR},
    batch_size=8,
    num_cpus=1,
    compute=ray.data.ActorPoolStrategy(size=2),
    batch_format="numpy",
).take_all()

# What did each actor return? Show the schema, vector geometry, and a sample.
sample = rows[0]
emb = np.asarray(sample["embedding"])
print(f"Rows embedded   : {len(rows)}")
print(f"Per-row keys    : {list(sample.keys())}")
print(f"Embedding shape : {emb.shape}    dtype: {emb.dtype}")
print(f"Vector L2 norm  : {float(np.linalg.norm(emb)):.3f}  "
      f"(SentenceTransformer default — not L2-normalised)")
print()
print(f"Sample → {sample['name']!r}  [{sample['category']}]")
print(f"  emb[:8] = {emb[:8].round(3).tolist()}")

In [ ]:
# Persist vectors and metadata as two separate files.
#
# The dense (N, 384) matrix goes to .npy for fast mmap-style loads at serve
# time; product name/category/id go to a small JSON sidecar so the serving
# layer never has to pull full product text into memory at query time.
from utils.embedding import save_embeddings_and_metadata

embeddings, metadata = save_embeddings_and_metadata(rows, EMBEDDINGS_PATH, METADATA_PATH)
print(f"Embeddings : {embeddings.shape}  →  {EMBEDDINGS_PATH}")
print(f"Metadata   : {len(metadata)} products  →  {METADATA_PATH}")


In [ ]:
import json

import numpy as np

from utils.viz import plot_similarity_heatmap

embeddings = np.load(EMBEDDINGS_PATH)
with open(METADATA_PATH) as f:
    metadata = json.load(f)

print(f"Embeddings shape : {embeddings.shape}")
print(f"Products indexed : {len(metadata)}")

plot_similarity_heatmap(embeddings, [m["name"] for m in metadata])

In [ ]:
from utils.viz import plot_tsne_comparison

base_embs, ft_embs, aligned_meta = plot_tsne_comparison(
    metadata, embeddings, catalog_records=catalog_records
)

In [ ]:
# Numeric check: did fine-tuning actually make the geometry category-aware?
#
# Simple "top-k neighbors share the same category" scores read ~100% on this
# demo because product names already sound like their category — so the base
# model looks strong before we even train. The two metrics below still move
# after fine-tuning and tell us the embedding space really did reshape.
from utils.viz import print_embedding_quality_report

print_embedding_quality_report(base_embs, ft_embs, aligned_meta)

---
## Stage 4 — Ray Serve: Online Recommendation API

**The problem this solves:** BLIP (224M params) and MiniLM (22M params) have very different latency, throughput, and resource profiles. Bundling them into one process means the slower model throttles the faster one.

Ray Serve deploys them as **independent deployments** that auto-scale separately. `ImageToText` can scale to 4 replicas for a spike in image uploads without adding MiniLM replicas you don't need. Both can be upgraded or rolled back independently, and the composing `RecommendationService` wires them together asynchronously behind a single `/recommend` endpoint.

In [ ]:
from ray import serve

from serve_app import build_app

# Pass paths through bind() — each replica gets them on construction, so we
# don't have to mutate os.environ on the driver.
app = build_app(
    embedding_model_dir=MODEL_OUTPUT_DIR,
    embeddings_path=EMBEDDINGS_PATH,
    metadata_path=METADATA_PATH,
)

serve.run(app)
print("Service running at http://localhost:8000")

In [ ]:
# Smoke-test the running /recommend endpoint with a known product image.
# Wireless Headphones is in Electronics — a healthy fine-tune should put
# other Electronics in the top results, not Books or Sports.
from utils import PRODUCTS, get_product_image, post_recommend

product = PRODUCTS[0]
img_arr = get_product_image(product)
result  = post_recommend(img_arr)

print(f"Caption : {result['caption']!r}")
print(f"\nTop-{len(result['recommendations'])} recommendations:")
for r in result["recommendations"]:
    print(
        f"  {r['rank']}. [{r['category']:18s}] {r['name']:35s}  sim={r['similarity']:.3f}"
    )


In [ ]:
# One image per category — does the pipeline generalise beyond headphones?
from utils import post_recommend

print(f"{'Category':<16} {'Caption':<45} {'Top recommendation'}")
print("-" * 85)
for cat in CATEGORIES:
    prod = next(p for p in PRODUCTS if p["category"] == cat)
    r = post_recommend(get_product_image(prod))
    top = r["recommendations"][0]
    print(
        f"{cat:<16} {r['caption'][:44]:<45} {top['name'][:20]} ({top['similarity']:.2f})"
    )


In [ ]:
# Teardown (optional — keep the service running if you want to use the Streamlit app)
# serve.shutdown()